# Multiple-model comparison: PC alpha sweep


## Motivation

Suppose we want to compare several outputs of a structure-learning algorithm produced under different hyperparameters — e.g. PC at multiple `alpha` values. We want to characterise complexity patterns and similarity patterns across the resulting DAG family.

In [1]:
"""Setup helpers for the bnmetrics example notebooks.

These notebooks are intentionally self-contained: they use small local
utilities (random DAG generator, simple synthetic-data sampler, basic
PC-CPDAG construction) so a reader can run them with only ``bnmetrics`` and
its optional ``[viz]`` extra installed. In production, prefer
``dagsampler`` for DAG construction and synthetic data and ``cbcd`` for
the PC algorithm and CPDAG output.
"""

import bnmetrics
import numpy as np
import pandas as pd
import random
from collections import deque

# Configure plotly so figures render in any notebook viewer (JupyterLab,
# classic Notebook, nbviewer, GitHub's static renderer) instead of only
# emitting a vnd.plotly.v1+json mimetype.
import plotly.io as pio
pio.renderers.default = "notebook_connected+plotly_mimetype"


# ---- DAG / data generation (would normally come from dagsampler) ----

def random_dag(n_nodes=40, edge_prob=0.1, seed=None):
    """Return a bnmetrics.GraphLike random DAG with X_1..X_n nodes."""
    rng = random.Random(seed)
    names = tuple(f"X_{i+1}" for i in range(n_nodes))
    topo = list(range(n_nodes))
    rng.shuffle(topo)
    endpoints = np.zeros((n_nodes, n_nodes), dtype=np.int8)
    for i in range(n_nodes):
        for j in range(i + 1, n_nodes):
            if rng.random() < edge_prob:
                src, dst = topo[i], topo[j]
                endpoints[src, dst] = bnmetrics.EndpointMark.ARROW
                endpoints[dst, src] = bnmetrics.EndpointMark.TAIL
    return bnmetrics.to_graphlike(endpoints, var_names=names)


def generate_data(g, n_samples=1000, stdev=1.0, seed=None):
    """Linear-Gaussian SCM sampling. Returns a pandas DataFrame
    with columns matching g.var_names."""
    rng = np.random.default_rng(seed)
    n = g.n_vars
    names = list(g.var_names) if g.var_names else [str(i) for i in range(n)]
    arr = np.array(g.endpoints)
    children: list[list[int]] = [[] for _ in range(n)]
    in_deg = [0] * n
    for i in range(n):
        for j in range(n):
            if (
                arr[i, j] == bnmetrics.EndpointMark.ARROW
                and arr[j, i] == bnmetrics.EndpointMark.TAIL
            ):
                children[i].append(j)
                in_deg[j] += 1
    queue = deque(v for v in range(n) if in_deg[v] == 0)
    topo: list[int] = []
    while queue:
        v = queue.popleft()
        topo.append(v)
        for c in children[v]:
            in_deg[c] -= 1
            if in_deg[c] == 0:
                queue.append(c)
    data = pd.DataFrame(index=range(n_samples), columns=names, dtype=float)
    for v in topo:
        parents = [
            i for i in range(n)
            if arr[i, v] == bnmetrics.EndpointMark.ARROW
            and arr[v, i] == bnmetrics.EndpointMark.TAIL
        ]
        if not parents:
            data[names[v]] = rng.standard_normal(n_samples)
        else:
            weights = rng.uniform(0.5, 1.5, size=len(parents))
            x = rng.normal(0, stdev, size=n_samples)
            for p, w in zip(parents, weights):
                x = x + w * data[names[p]].values
            data[names[v]] = x
    return data


# ---- adjacency-matrix helpers ---------------------------------------

def from_01_adj(adj, var_names=None):
    """Convert a {0,1} adjacency matrix (1 = directed edge i→j) to a
    bnmetrics.GraphLike. If both adj[i,j] == 1 and adj[j,i] == 1, the edge
    is treated as undirected (CPDAG-style)."""
    a = np.asarray(adj)
    n = a.shape[0]
    endpoints = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        for j in range(i + 1, n):
            ij, ji = int(a[i, j]), int(a[j, i])
            if ij == 1 and ji == 1:
                endpoints[i, j] = endpoints[j, i] = bnmetrics.EndpointMark.TAIL
            elif ij == 1:
                endpoints[i, j] = bnmetrics.EndpointMark.ARROW
                endpoints[j, i] = bnmetrics.EndpointMark.TAIL
            elif ji == 1:
                endpoints[i, j] = bnmetrics.EndpointMark.TAIL
                endpoints[j, i] = bnmetrics.EndpointMark.ARROW
    names = tuple(var_names) if var_names is not None else None
    return bnmetrics.to_graphlike(endpoints, var_names=names)


def dag_to_cpdag(g):
    """Convert a DAG to its CPDAG (without Meek-rule closure — same
    semantics as 0.1.x's `dag_to_cpdag`). For a Meek-closed CPDAG,
    use cbcd's `DAG.to_cpdag()` instead.
    """
    g = bnmetrics.to_graphlike(g)
    n = g.n_vars
    arr = np.array(g.endpoints)
    in_collider = np.zeros((n, n), dtype=bool)
    for v in range(n):
        parents = [
            u for u in range(n)
            if arr[u, v] == bnmetrics.EndpointMark.ARROW
            and arr[v, u] == bnmetrics.EndpointMark.TAIL
        ]
        for ip in range(len(parents)):
            for jp in range(ip + 1, len(parents)):
                u, w = parents[ip], parents[jp]
                if (
                    arr[u, w] == bnmetrics.EndpointMark.NO_EDGE
                    and arr[w, u] == bnmetrics.EndpointMark.NO_EDGE
                ):
                    in_collider[u, v] = True
                    in_collider[w, v] = True
    cpdag = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if (
                arr[i, j] == bnmetrics.EndpointMark.ARROW
                and arr[j, i] == bnmetrics.EndpointMark.TAIL
            ):
                if in_collider[i, j]:
                    cpdag[i, j] = bnmetrics.EndpointMark.ARROW
                    cpdag[j, i] = bnmetrics.EndpointMark.TAIL
                else:
                    cpdag[i, j] = bnmetrics.EndpointMark.TAIL
                    cpdag[j, i] = bnmetrics.EndpointMark.TAIL
    return bnmetrics.to_graphlike(cpdag, var_names=g.var_names)


# ---- perturbation helper for "fake learned graph" demos ----------

def perturb(g, n_drops=0, n_adds=0, n_reverses=0, seed=None):
    """Return a perturbed copy of g: drop / add / reverse the requested
    number of directed edges. Used as a stand-in for a real causal-
    discovery output when we don't want to depend on a PC algorithm.

    For real workflows, swap this for `cbcd.pc(data, ...)` or any
    PC implementation whose output you can wrap in `from_01_adj()`.
    """
    g = bnmetrics.to_graphlike(g)
    rng = random.Random(seed)
    arr = np.array(g.endpoints).copy()
    n = arr.shape[0]
    directed_edges = [
        (i, j) for i in range(n) for j in range(n)
        if arr[i, j] == bnmetrics.EndpointMark.ARROW
        and arr[j, i] == bnmetrics.EndpointMark.TAIL
    ]
    rng.shuffle(directed_edges)
    # Drops
    for k in range(min(n_drops, len(directed_edges))):
        i, j = directed_edges[k]
        arr[i, j] = arr[j, i] = bnmetrics.EndpointMark.NO_EDGE
    remaining = directed_edges[n_drops:]
    # Reverses
    for k in range(min(n_reverses, len(remaining))):
        i, j = remaining[k]
        arr[i, j] = bnmetrics.EndpointMark.TAIL
        arr[j, i] = bnmetrics.EndpointMark.ARROW
    # Adds (only between currently-non-adjacent pairs to keep it a DAG-ish)
    non_edges = [
        (i, j) for i in range(n) for j in range(i + 1, n)
        if arr[i, j] == bnmetrics.EndpointMark.NO_EDGE
    ]
    rng.shuffle(non_edges)
    for k in range(min(n_adds, len(non_edges))):
        i, j = non_edges[k]
        # Random direction
        if rng.random() < 0.5:
            arr[i, j] = bnmetrics.EndpointMark.ARROW
            arr[j, i] = bnmetrics.EndpointMark.TAIL
        else:
            arr[i, j] = bnmetrics.EndpointMark.TAIL
            arr[j, i] = bnmetrics.EndpointMark.ARROW
    return bnmetrics.to_graphlike(arr, var_names=g.var_names)


# ---- viz helper: 0.1.x's compare_two_bn(option=2) ----------------

def mb_pair(g1, g2, var):
    """Return ``(g1.MB(var), g2 restricted to g1.MB(var) indices)``.
    Useful for side-by-side rendering of "true MB" vs "estimated MB
    over the same nodes." Replaces 0.1.x's ``compare_two_bn(option=2)``."""
    g1n, g2n = bnmetrics.to_graphlike(g1), bnmetrics.to_graphlike(g2)
    idx = bnmetrics.markov_blanket_indices(g1n, var)
    sub_g1 = bnmetrics.markov_blanket(g1n, var)
    arr2 = g2n.endpoints[np.ix_(idx, idx)]
    sub_names = (
        tuple(g2n.var_names[i] for i in idx) if g2n.var_names else None
    )
    sub_g2 = bnmetrics.to_graphlike(arr2, var_names=sub_names)
    return sub_g1, sub_g2


print("bnmetrics:", bnmetrics.__version__)


bnmetrics: 0.2.0.dev0


### Data generation

Random 40-node DAG with 10% density, then linear-Gaussian data over it.

In [2]:
true_dag = random_dag(n_nodes=40, edge_prob=0.1, seed=55)
data = generate_data(true_dag, n_samples=1000, stdev=1.0, seed=55)
f'true DAG: {true_dag.n_vars} vars, {bnmetrics.count_edges(true_dag)} edges; data: {data.shape}'

'true DAG: 40 vars, 76 edges; data: (1000, 40)'

## Structure learning (simulated alpha sweep)

We simulate the effect of varying PC's alpha threshold by fabricating 20 graphs with progressively more edges (smaller alpha → fewer edges retained → looks like dropping). For a real PC sweep, swap the perturbation loop for `cbcd.pc(data, alpha=a, ...)` calls — the rest of this notebook is unchanged.

In [3]:
alphas = np.linspace(0.01, 0.2, 20)
# Simulate: lower alpha drops more truth-edges; higher alpha
# adds more spurious edges. Reverses kept constant.
list_of_models = []
n_truth_edges = bnmetrics.count_edges(true_dag)
for k, a in enumerate(alphas):
    n_drops = int(round((1 - a / 0.2) * 0.4 * n_truth_edges))
    n_adds = int(round((a / 0.2) * 0.3 * n_truth_edges))
    list_of_models.append(
        perturb(true_dag, n_drops=n_drops, n_adds=n_adds,
                n_reverses=2, seed=100 + k)
    )
len(list_of_models)

20

## Complexity patterns and algorithm behavior

`bnmetrics.compare_models_descriptive` plots each descriptive metric (n_edges, n_colliders, etc.) as model_name → value, one panel per metric. With `per_node=True`, a Plotly dropdown lets you switch between whole-graph and per-MB views.

In [4]:
model_labels = [f'{a:.3f}' for a in alphas]

bnmetrics.compare_models_descriptive(
    list_of_models,
    model_labels,
    descriptive=['n_edges', 'n_colliders', 'n_directed_arcs',
                 'n_undirected_arcs', 'n_root_nodes', 'n_leaf_nodes',
                 'n_isolated_nodes', 'n_reversible_arcs'],
    per_node=True,
    title='PC alpha sweep — descriptive metrics',
)

As expected, higher alpha produces denser networks (more edges, more colliders, more directed arcs).

Local view: zoom into a specific Markov blanket (e.g. `X_1`) by selecting it in the dropdown. To compare two specific models on that MB, use `plot_side_by_side` directly.

In [5]:
g_low = list_of_models[6]
g_high = list_of_models[13]
sub_low, sub_high = mb_pair(g_low, g_high, 'X_1')
bnmetrics.plot_side_by_side(
    sub_low, sub_high,
    name1=f'alpha={alphas[6]:.3f}',
    name2=f'alpha={alphas[13]:.3f}',
    highlight_nodes=['X_1'],
    direction='auto',
)

## Similarity patterns and model stability

`bnmetrics.compare_models_comparative` computes one comparative metric across all (n × n) pairs and renders the heatmap. F1 of 1.0 means two models produce identical structure; lower F1 means the structures diverge.

In [6]:
bnmetrics.compare_models_comparative(
    list_of_models,
    model_labels,
    metric='f1',
    per_node=True,
    title='PC alpha sweep — pairwise F1',
)

In the dropdown, switch to a specific MB (e.g. `X_1`) — you'll often see clusters of identical models for the same local structure even when the global F1 differs.

## Conclusion

Two views answer most questions about a model family:

- `compare_models_descriptive`: how complexity changes across hyperparameters.
- `compare_models_comparative`: how similar the models are to each other (and to the truth, if included).

### Other Cases:
- [Evaluate Single DAG](./evaluate_single_dag.ipynb)
- [Compare Two DAGs](./compare_two_dags.ipynb)
- [SID](./sid.ipynb)